# EXP21E-v7 — End-to-End Tabular MLP Baseline

Runs the leakage-safe non-graph baseline over five predeclared seeds. Checkpoint and threshold selection use validation only.


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21E"
  version: "v7"
  window_id: "primary_2018"
  seed: 42
  evaluation_seed: 1042
  audit_seed: 2042
  status: "tabular_mlp_baseline"

data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

features:
  mode: "clean" # "clean" (tanpa Zip & Merchant City demi anti-leakage)
  use_cyclical_time: true
  use_amount_log: true
  exclude_high_cardinality: true

sampling:
  method: "uniform" # Pilihan: "uniform", "topology", "importance"
  undersampling_ratio: 1.0
  weight_attr: "sampling_weight"
  smoothing_alpha: 0.1

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

classifier:
  type: "end_to_end_tabular_mlp"

threshold_calibration:
  selection_metric: "val_f1"
  min_precision_constraint: 0.15

reproducibility:
  deterministic: true
  warn_only: true
  compute_dataset_sha256: false

validation:
  empirical_draws: 10000
  max_probability_error: 0.03

causal_evaluation:
  protocol: "daily_batch_frozen_history"
  cutoff: "start_of_day"
  history_inclusive: false
  add_completed_day_after_inference: true
  use_labels_in_graph_state: false

smoke:
  max_rows_per_split: 20000
  epochs: 2
  max_days_per_eval_split: 2

output:
  result_dir: "/kaggle/working/result/exp21e"
  snapshot_source: true


In [ ]:
%%writefile config.py
import yaml
from pathlib import Path


def load_config(config_path: str = "config.reference.yaml") -> dict:
    """Memuat dan memvalidasi konfigurasi eksperimen 21E."""
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Berkas konfigurasi tidak ditemukan: {config_path}")

    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    return config


In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler


def load_and_split_raw_data(
    file_path: str,
    start_year: int = 2016,
    nrows: int = None,
    reference_date: str = "2018-01-01",
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
) -> tuple:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter era modern (Year >= start_year),
    melakukan parsing datetime kronologis, dan memotongnya ke dalam 4 partisi temporal SHINE FRAUD II
    SEBELUM proses rekayasa fitur atau penskalaan (Strict Out-of-Sample Protocol).
    """
    print(
        f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})..."
    )

    # 1. Pemuatan chunked efisien jika start_year ditentukan
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000

        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk["Year"] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break

        if not chunks:
            raise ValueError(
                f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}"
            )

        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data mentah...")
    # Deterministic identifier within the filtered source. It survives sorting and
    # is stored with predictions so paired sampler outputs can be aligned safely.
    df["transaction_id"] = np.arange(len(df), dtype=np.int64)
    print("Parsing timestamps...")
    df["Time"] = df["Time"].fillna("00:00")
    df["datetime"] = pd.to_datetime(
        df["Year"].astype(str)
        + "-"
        + df["Month"].astype(str).str.zfill(2)
        + "-"
        + df["Day"].astype(str).str.zfill(2)
        + " "
        + df["Time"]
    )

    print("Sorting chronologically...")
    df = df.sort_values("datetime").reset_index(drop=True)

    print("Cleaning Amount & Target Label...")
    df["Amount"] = (
        df["Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )
    df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

    # 2. Pembagian Jendela Waktu Temporal (SHINE FRAUD II)
    print("\nMelakukan pembagian temporal window strictly chronologic...")
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)

    train_df = (
        df[(df["datetime"] >= t0) & (df["datetime"] < t_val_start)]
        .copy()
        .reset_index(drop=True)
    )
    val_df = (
        df[(df["datetime"] >= t_val_start) & (df["datetime"] < t_gap_start)]
        .copy()
        .reset_index(drop=True)
    )
    gap_df = (
        df[(df["datetime"] >= t_gap_start) & (df["datetime"] < t_test_start)]
        .copy()
        .reset_index(drop=True)
    )
    test_df = (
        df[(df["datetime"] >= t_test_start) & (df["datetime"] < t_test_end)]
        .copy()
        .reset_index(drop=True)
    )

    boundaries = {
        "train": (train_df, t0, t_val_start),
        "validation": (val_df, t_val_start, t_gap_start),
        "embargo_gap": (gap_df, t_gap_start, t_test_start),
        "test": (test_df, t_test_start, t_test_end),
    }
    for split_name, (split_df, lower, upper) in boundaries.items():
        if len(split_df) and not (
            (split_df["datetime"] >= lower).all()
            and (split_df["datetime"] < upper).all()
        ):
            raise AssertionError(f"Temporal boundary violation in split: {split_name}")

    def compute_split_stats(
        name: str,
        split_df: pd.DataFrame,
        start_dt,
        end_dt,
        include_label_stats: bool = True,
    ):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0 if include_label_stats else None,
                "fraud_rate_pct": 0.0 if include_label_stats else None,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0,
            }
        n_fraud = int(split_df["Is Fraud?"].sum()) if include_label_stats else None
        n_users = int(split_df["User"].nunique())
        n_merchants = int(split_df["Merchant Name"].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4)
            if include_label_stats
            else None,
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants,
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)

    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment (EXP21E Pure End-to-End)",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days,
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats(
                "Validation", val_df, t_val_start, t_gap_start
            ),
            "embargo_gap": compute_split_stats(
                "Embargo Gap", gap_df, t_gap_start, t_test_start
            ),
            # Test-label summaries are intentionally deferred until the frozen
            # checkpoint and validation threshold are evaluated in main.py.
            "test": compute_split_stats(
                "Test", test_df, t_test_start, t_test_end, include_label_stats=False
            ),
            "total_window": compute_split_stats(
                "Total Window",
                total_window_df,
                t0,
                t_test_end,
                include_label_stats=False,
            ),
        },
    }

    return train_df, val_df, test_df, gap_df, stats_dict


def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True,
    use_amount_log: bool = True,
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol (EXP21E):
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df, test_df, gap_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang bebas memorization.
    5. Menambahkan amount_log (log1p Amount) untuk menstabilkan distribusi numerik.
    """
    print(
        f"\n[STRICT ANTI-LEAKAGE] Menerapkan preprocessing (Mode: {feature_mode}, Cyclical Time: {use_cyclical_time}, Amount Log: {use_amount_log})..."
    )

    # Simpan nilai Amount mentah positif untuk log amount
    for df_split in [train_df, val_df, test_df] + (
        [gap_df] if gap_df is not None and len(gap_df) > 0 else []
    ):
        if use_amount_log:
            df_split["amount_log"] = np.log1p(
                np.maximum(0.0, df_split["Amount"].values)
            ).astype(np.float32)

    # Tentukan atribut kategorikal yang diizinkan
    if feature_mode == "clean":
        cat_columns = ["Use Chip", "Merchant State", "MCC", "Errors?"]
        print("  • Fitur High-Cardinality ('Zip', 'Merchant City') DIHAPUS dari input!")
    elif feature_mode == "clean_basic":
        cat_columns = ["Use Chip", "MCC", "Errors?"]
        print(f"  • Menggunakan fitur kategorikal minimal: {cat_columns}")
    else:
        raise ValueError(
            f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean' atau 'clean_basic'."
        )

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN
    scaler_amount = StandardScaler()
    train_df["Amount"] = scaler_amount.fit_transform(train_df[["Amount"]])
    val_df["Amount"] = scaler_amount.transform(val_df[["Amount"]])
    test_df["Amount"] = scaler_amount.transform(test_df[["Amount"]])
    if gap_df is not None and len(gap_df) > 0:
        gap_df["Amount"] = scaler_amount.transform(gap_df[["Amount"]])

    print(
        f"  • Amount Scaler fit pada Train: mean={scaler_amount.mean_[0]:.4f}, std={scaler_amount.scale_[0]:.4f}"
    )

    if use_amount_log:
        scaler_log = StandardScaler()
        train_df["amount_log"] = scaler_log.fit_transform(train_df[["amount_log"]])
        val_df["amount_log"] = scaler_log.transform(val_df[["amount_log"]])
        test_df["amount_log"] = scaler_log.transform(test_df[["amount_log"]])
        if gap_df is not None and len(gap_df) > 0:
            gap_df["amount_log"] = scaler_log.transform(gap_df[["amount_log"]])
        print(
            f"  • Amount Log Scaler fit pada Train: mean={scaler_log.mean_[0]:.4f}, std={scaler_log.scale_[0]:.4f}"
        )

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna("<missing>").astype(str)
        val_df[col] = val_df[col].fillna("<missing>").astype(str)
        test_df[col] = test_df[col].fillna("<missing>").astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna("<missing>").astype(str)

        # Pelajari mapping unik dari TRAIN saja
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai dari index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id

        # Transform dengan fallback OOV = 0
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)

        num_oov_val = int((val_df[col] == 0).sum())
        num_oov_test = int((test_df[col] == 0).sum())
        print(
            f"  • Kolom '{col}': {len(cat_to_id)} kategori train dipelajari (OOV di Val: {num_oov_val}, di Test: {num_oov_test})"
        )

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ["Amount"]
    if use_amount_log:
        feature_cols.append("amount_log")
    feature_cols = feature_cols + cat_columns

    if use_cyclical_time:
        all_splits = [train_df, val_df, test_df]
        if gap_df is not None and len(gap_df) > 0:
            all_splits.append(gap_df)

        for split_df in all_splits:
            hour_float = (
                split_df["datetime"].dt.hour + split_df["datetime"].dt.minute / 60.0
            )
            split_df["hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0).astype(
                np.float32
            )
            split_df["hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0).astype(
                np.float32
            )

            dow = split_df["datetime"].dt.dayofweek
            split_df["dow_sin"] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df["dow_cos"] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
        feature_cols = feature_cols + time_cols
        print(f"  • Ditambahkan 4 fitur waktu siklik: {time_cols}")

    print(
        f"Total fitur transaksi bersih (feature_cols): {len(feature_cols)} fitur -> {feature_cols}"
    )

    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode,
    }

    return train_df, val_df, test_df, gap_df, feature_cols, artifacts


In [ ]:
%%writefile model.py
from __future__ import annotations

import torch
import torch.nn as nn


class TabularMLP(nn.Module):
    def __init__(
        self, input_dim: int, hidden_channels: int = 128, dropout: float = 0.2
    ):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features).squeeze(-1)


def manual_weighted_bce(logits: torch.Tensor, targets: torch.Tensor, pos_weight: float):
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (1 + (pos_weight - 1) * targets) * (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()


def manual_unweighted_bce(logits: torch.Tensor, targets: torch.Tensor):
    max_val = torch.clamp(-logits, min=0)
    loss = (
        (1 - targets) * logits
        + max_val
        + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()


In [ ]:
%%writefile reproducibility.py
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Iterator

import numpy as np
import torch


def seed_everything(
    seed: int, deterministic: bool = True, warn_only: bool = True
) -> None:
    """Seed all RNGs used by the EXP21E-v7 pipeline."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=warn_only)


@contextmanager
def temporary_seed(seed: int) -> Iterator[None]:
    """Temporarily reset RNGs so validation/test sampling is repeatable."""
    python_state = random.getstate()
    numpy_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    seed_everything(seed)
    try:
        yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        torch.random.set_rng_state(torch_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)


def stable_hash(value: Any, length: int = 12) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode(
        "utf-8"
    )
    return hashlib.sha256(payload).hexdigest()[:length]


def sha256_file(path: str | Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def file_fingerprint(path: str | Path, with_sha256: bool = False) -> dict[str, Any]:
    file_path = Path(path).resolve()
    stat = file_path.stat()
    output: dict[str, Any] = {
        "path": str(file_path),
        "size_bytes": stat.st_size,
        "modified_time_ns": stat.st_mtime_ns,
    }
    if with_sha256:
        output["sha256"] = sha256_file(file_path)
    return output


def git_revision(workdir: str | Path) -> dict[str, Any]:
    root = Path(workdir).resolve()
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, text=True, stderr=subprocess.DEVNULL
        ).strip()
        dirty = bool(
            subprocess.check_output(
                ["git", "status", "--porcelain"],
                cwd=root,
                text=True,
                stderr=subprocess.DEVNULL,
            ).strip()
        )
        return {"commit": commit, "dirty": dirty}
    except (OSError, subprocess.CalledProcessError):
        return {"commit": None, "dirty": None}


def environment_manifest() -> dict[str, Any]:
    try:
        import torch_geometric

        pyg_version = torch_geometric.__version__
    except Exception:
        pyg_version = None

    try:
        import pyg_lib

        pyg_lib_version = getattr(pyg_lib, "__version__", "installed")
    except Exception:
        pyg_lib_version = None

    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "torch": torch.__version__,
        "torch_geometric": pyg_version,
        "pyg_lib": pyg_lib_version,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None,
    }


def snapshot_source(source_dir: str | Path, destination: str | Path) -> None:
    """Copy the exact text/code assets used by a run into its result directory."""
    source = Path(source_dir).resolve()
    target = Path(destination).resolve()
    target.mkdir(parents=True, exist_ok=True)
    allowed_suffixes = {".py", ".yaml", ".yml", ".json", ".md", ".ipynb"}

    for item in source.iterdir():
        if item.name == "__pycache__":
            continue
        if item.is_file() and item.suffix.lower() in allowed_suffixes:
            shutil.copy2(item, target / item.name)


In [ ]:
%%writefile tabular_validation.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from data_preprocessing import apply_strict_anti_leakage_preprocessing
from model import TabularMLP


EXPECTED_FEATURES = {
    "Amount",
    "amount_log",
    "Use Chip",
    "Merchant State",
    "MCC",
    "Errors?",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
}
FORBIDDEN_FEATURES = {"User", "Merchant Name", "Zip", "Merchant City", "Is Fraud?"}


def toy_split(start: str, rows: int, unseen: bool = False) -> pd.DataFrame:
    index = np.arange(rows)
    return pd.DataFrame(
        {
            "transaction_id": index,
            "datetime": pd.date_range(start, periods=rows, freq="h"),
            "Amount": 10.0 + index.astype(float),
            "Use Chip": "Online Transaction"
            if unseen
            else np.where(index % 2, "Chip Transaction", "Swipe Transaction"),
            "Merchant State": "ZZ" if unseen else np.where(index % 2, "CA", "NY"),
            "MCC": "9999" if unseen else np.where(index % 2, "5411", "5812"),
            "Errors?": "New Error" if unseen else np.where(index % 3, None, "Bad PIN"),
            "User": [f"u{i % 3}" for i in index],
            "Merchant Name": [f"m{i % 4}" for i in index],
            "Zip": 10000 + index,
            "Merchant City": "city",
            "Is Fraud?": (index % 4 == 0).astype(int),
        }
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Validate the EXP21E-v7 tabular baseline contract"
    )
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    train = toy_split("2018-01-01", 24)
    validation = toy_split("2018-02-01", 8, unseen=True)
    test = toy_split("2018-03-01", 8, unseen=True)
    gap = toy_split("2018-02-15", 4, unseen=True)
    transformed = apply_strict_anti_leakage_preprocessing(
        train.copy(), validation.copy(), test.copy(), gap.copy()
    )
    train_t, val_t, test_t, gap_t, features, artifacts = transformed

    checks = {
        "exact_feature_contract": set(features) == EXPECTED_FEATURES,
        "no_identity_or_target_features": not FORBIDDEN_FEATURES.intersection(features),
        "validation_oov_maps_to_zero": bool((val_t["Merchant State"] == 0).all()),
        "test_oov_maps_to_zero": bool((test_t["Merchant State"] == 0).all()),
        "gap_oov_maps_to_zero": bool((gap_t["Merchant State"] == 0).all()),
        "scaler_fit_on_train_only": bool(abs(float(train_t["Amount"].mean())) < 1e-6),
        "no_graph_features": not any(
            "degree" in name or "graph" in name for name in features
        ),
    }
    model = TabularMLP(input_dim=len(features), hidden_channels=128, dropout=0.2)
    output = model(torch.tensor(train_t[features].to_numpy(), dtype=torch.float32))
    checks["model_output_shape"] = list(output.shape) == [len(train_t)]
    checks["architecture_has_two_batchnorm_layers"] = (
        sum(isinstance(module, torch.nn.BatchNorm1d) for module in model.modules()) == 2
    )
    report = {
        "status": "passed" if all(checks.values()) else "failed",
        "checks": checks,
        "feature_columns": features,
        "train_only_categories": {
            name: len(mapping)
            for name, mapping in artifacts["categorical_encoders"].items()
        },
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    }
    text = json.dumps(report, indent=2)
    print(text)
    if args.output:
        output_path = Path(args.output)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(text + "\n", encoding="utf-8")
    if report["status"] != "passed":
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import gc
import json
import platform
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import yaml
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, TensorDataset

from config import load_config
from data_preprocessing import (
    apply_strict_anti_leakage_preprocessing,
    load_and_split_raw_data,
)
from model import TabularMLP, manual_unweighted_bce, manual_weighted_bce
from reproducibility import (
    environment_manifest,
    file_fingerprint,
    git_revision,
    seed_everything,
    snapshot_source,
    stable_hash,
    temporary_seed,
)


FORBIDDEN_FEATURES = {"User", "Merchant Name", "Zip", "Merchant City", "Is Fraud?"}


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def peak_cpu_memory_bytes() -> int | None:
    try:
        import resource

        peak = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return peak * 1024 if platform.system() != "Darwin" else peak
    except (ImportError, OSError):
        return None


def find_optimal_threshold(
    probabilities: np.ndarray, labels: np.ndarray, min_precision: float
):
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    if thresholds.size == 0:
        return 0.5, {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0,
            "candidate_thresholds": 0,
            "constraint_applied": False,
        }
    p = precision[:-1]
    r = recall[:-1]
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    eligible = p >= min_precision
    candidates = (
        np.flatnonzero(eligible) if eligible.any() else np.arange(len(thresholds))
    )
    best = candidates[np.argmax(f1[candidates])]
    return float(thresholds[best]), {
        "tau": float(thresholds[best]),
        "val_f1": float(f1[best]),
        "val_precision": float(p[best]),
        "val_recall": float(r[best]),
        "candidate_thresholds": int(len(thresholds)),
        "constraint_applied": bool(eligible.any()),
        "min_precision_constraint": float(min_precision),
    }


def evaluate_predictions(
    probabilities: np.ndarray, labels: np.ndarray, threshold: float
) -> dict[str, Any]:
    predictions = (probabilities >= threshold).astype(int)
    default_predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    recall = float(recall_score(labels, predictions, zero_division=0))
    precision = float(precision_score(labels, predictions, zero_division=0))
    specificity = float(tn / (tn + fp)) if (tn + fp) else 0.0
    total_positive = int(labels.sum())
    recall_at_k: dict[str, Any] = {}
    for requested in (100, 200, 500, 1000):
        actual = min(requested, len(labels))
        indices = np.argsort(probabilities)[-actual:]
        captured = int(labels[indices].sum())
        recall_at_k[str(requested)] = {
            "actual_k": actual,
            "true_positives": captured,
            "recall": float(captured / total_positive) if total_positive else 0.0,
            "precision": float(captured / actual) if actual else 0.0,
        }
    k_1pct = max(1, int(np.ceil(0.01 * len(labels))))
    top = np.argsort(probabilities)[-k_1pct:]
    baseline = float(labels.mean()) if len(labels) else 0.0
    selected = float(labels[top].mean()) if len(top) else 0.0
    return {
        "metrics_optimal_threshold": {
            "threshold": threshold,
            "f1_score": float(f1_score(labels, predictions, zero_division=0)),
            "f1_macro": float(
                f1_score(labels, predictions, average="macro", zero_division=0)
            ),
            "recall": recall,
            "precision": precision,
            "accuracy": float(accuracy_score(labels, predictions)),
            "gmean": float(np.sqrt(recall * specificity)),
            "alerts_per_captured_fraud": float((tp + fp) / tp) if tp else None,
            "confusion_matrix": {
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            },
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": float(f1_score(labels, default_predictions, zero_division=0)),
            "recall": float(recall_score(labels, default_predictions, zero_division=0)),
            "precision": float(
                precision_score(labels, default_predictions, zero_division=0)
            ),
        },
        "metrics_threshold_independent": {
            "auprc": float(average_precision_score(labels, probabilities)),
            "roc_auc": float(roc_auc_score(labels, probabilities)),
            "lift_1_percent": float(selected / baseline) if baseline else 1.0,
            "recall_at_k": recall_at_k,
        },
    }


def smoke_subset(frame: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if len(frame) <= max_rows:
        return frame.copy().reset_index(drop=True)
    positive = frame[frame["Is Fraud?"] == 1].iloc[: max_rows // 4]
    negative = frame[frame["Is Fraud?"] == 0].sample(
        n=max_rows - len(positive), random_state=seed
    )
    return (
        pd.concat([positive, negative])
        .sort_values(["datetime", "transaction_id"])
        .reset_index(drop=True)
    )


def make_loader(
    frame: pd.DataFrame, features: list[str], batch_size: int, shuffle: bool, seed: int
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(frame[features].to_numpy(), dtype=torch.float32),
        torch.tensor(frame["Is Fraud?"].to_numpy(), dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )


def infer(
    model: TabularMLP, loader: DataLoader, device: torch.device, seed: int
) -> dict[str, Any]:
    model.eval()
    probabilities: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    loss_sum = 0.0
    count = 0
    start = time.time()
    with temporary_seed(seed), torch.no_grad():
        for features, targets in loader:
            features = features.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            logits = model(features)
            loss_sum += float(manual_unweighted_bce(logits, targets).item()) * len(
                targets
            )
            count += len(targets)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(targets.cpu().numpy())
    elapsed = time.time() - start
    return {
        "probabilities": np.concatenate(probabilities),
        "labels": np.concatenate(labels),
        "loss": loss_sum / max(count, 1),
        "seconds": elapsed,
        "throughput_rows_per_second": count / elapsed if elapsed else None,
    }


def prediction_frame(
    frame: pd.DataFrame, probabilities: np.ndarray, split: str, threshold: float
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "split": split,
            "transaction_id": frame["transaction_id"].to_numpy(),
            "timestamp": frame["datetime"].astype(str).to_numpy(),
            "user_id": frame["User"].astype(str).to_numpy(),
            "merchant_id": frame["Merchant Name"].astype(str).to_numpy(),
            "label": frame["Is Fraud?"].astype(int).to_numpy(),
            "probability": probabilities,
            "threshold": threshold,
            "prediction": (probabilities >= threshold).astype(int),
        }
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description="EXP21E-v7 End-to-End Tabular MLP baseline"
    )
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--device", default=None)
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    cfg = load_config(args.config)
    experiment = cfg["experiment"]
    seed = int(args.seed if args.seed is not None else experiment.get("seed", 42))
    evaluation_seed = int(experiment.get("evaluation_seed", 1042))
    cfg["experiment"]["seed"] = seed
    seed_everything(
        seed,
        deterministic=cfg["reproducibility"].get("deterministic", True),
        warn_only=cfg["reproducibility"].get("warn_only", True),
    )
    device = torch.device(
        args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    )
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    version = "v7"
    config_hash = stable_hash({"config": cfg, "smoke": args.smoke})
    run_id = f"exp21e_{version}_{experiment.get('window_id', 'primary_2018')}_tabular_mlp_seed{seed}_{config_hash}"
    run_dir = Path(cfg["output"]["result_dir"]) / version / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    source_dir = Path(__file__).resolve().parent
    if cfg["output"].get("snapshot_source", True):
        snapshot_source(source_dir, run_dir / "code_snapshot")
    (run_dir / "resolved_config.yaml").write_text(
        yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8"
    )
    write_json(run_dir / "environment.json", environment_manifest())
    dataset_path = Path(cfg["data"]["transactions"])
    manifest = {
        "experiment_id": "EXP21E",
        "version": version,
        "stage": "tabular_mlp_baseline",
        "run_id": run_id,
        "model": "tabular_mlp",
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "window_id": experiment.get("window_id", "primary_2018"),
        "config_hash": config_hash,
        "smoke": args.smoke,
        "device": str(device),
        "git": git_revision(source_dir),
        "dataset": file_fingerprint(
            dataset_path,
            with_sha256=cfg["reproducibility"].get("compute_dataset_sha256", False),
        ),
        "status": "running",
    }
    write_json(run_dir / "manifest.json", manifest)
    started = time.time()
    time_cfg = cfg["time_window"]
    train_df, val_df, test_df, gap_df, statistics = load_and_split_raw_data(
        file_path=str(dataset_path),
        start_year=cfg["data"].get("start_year", 2016),
        nrows=cfg["data"].get("max_rows"),
        reference_date=time_cfg["reference_date"],
        train_days=time_cfg["train_days"],
        val_days=time_cfg["val_days"],
        gap_days=time_cfg["gap_days"],
        test_days=time_cfg["test_days"],
    )
    if args.smoke:
        limit = int(cfg["smoke"].get("max_rows_per_split", 20_000))
        train_df, val_df, test_df = (
            smoke_subset(train_df, limit, seed),
            smoke_subset(val_df, limit, seed + 1),
            smoke_subset(test_df, limit, seed + 2),
        )
        gap_df = smoke_subset(gap_df, min(limit, len(gap_df)), seed + 3)
    feature_cfg = cfg["features"]
    train_df, val_df, test_df, gap_df, features, _ = (
        apply_strict_anti_leakage_preprocessing(
            train_df,
            val_df,
            test_df,
            gap_df,
            feature_mode=feature_cfg["mode"],
            use_cyclical_time=feature_cfg["use_cyclical_time"],
            use_amount_log=feature_cfg["use_amount_log"],
        )
    )
    forbidden = sorted(FORBIDDEN_FEATURES.intersection(features))
    feature_audit = {
        "feature_columns": features,
        "forbidden_features": forbidden,
        "uses_graph_aggregate": False,
        "fit_scope": "train_only",
        "passed": not forbidden,
    }
    write_json(run_dir / "tabular_feature_audit.json", feature_audit)
    if not feature_audit["passed"]:
        raise RuntimeError(f"Forbidden MLP features: {forbidden}")
    data_seconds = time.time() - started
    training_cfg = cfg["training"]
    batch_size = int(training_cfg["batch_size"])
    train_loader = make_loader(train_df, features, batch_size, True, seed)
    val_loader = make_loader(val_df, features, batch_size, False, evaluation_seed)
    model = TabularMLP(
        len(features),
        int(cfg["model"]["hidden_channels"]),
        float(cfg["model"]["dropout"]),
    ).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    optimizer = optim.Adam(
        model.parameters(),
        lr=float(training_cfg["learning_rate"]),
        weight_decay=float(training_cfg["weight_decay"]),
    )
    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=float(training_cfg["lr_scheduler_factor"]),
            patience=int(training_cfg["lr_scheduler_patience"]),
        )
        if training_cfg.get("use_lr_scheduler", True)
        else None
    )
    negatives, positives = (
        int((train_df["Is Fraud?"] == 0).sum()),
        int((train_df["Is Fraud?"] == 1).sum()),
    )
    ratio = negatives / positives if positives else 1.0
    positive_weight = (
        min(np.sqrt(ratio), float(training_cfg.get("max_pos_weight", 50.0)))
        if training_cfg.get("loss_weight_strategy") == "sqrt"
        else ratio
    )
    epochs = int(cfg["smoke"]["epochs"] if args.smoke else training_cfg["epochs"])
    patience = int(training_cfg["early_stopping_patience"])
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_auprc": [],
        "val_f1": [],
    }
    best_auprc, best_epoch, best_state, counter = float("-inf"), 0, None, 0
    training_start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        correct = 0
        seen = 0
        for features_batch, targets in train_loader:
            features_batch, targets = (
                features_batch.to(device, non_blocking=True),
                targets.to(device, non_blocking=True),
            )
            optimizer.zero_grad(set_to_none=True)
            logits = model(features_batch)
            loss = manual_weighted_bce(logits, targets, positive_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), float(training_cfg["grad_clip_norm"])
            )
            optimizer.step()
            loss_sum += float(loss.item()) * len(targets)
            seen += len(targets)
            correct += int(
                ((torch.sigmoid(logits) >= 0.5).float() == targets).sum().item()
            )
        validation = infer(model, val_loader, device, evaluation_seed)
        val_auprc = float(
            average_precision_score(validation["labels"], validation["probabilities"])
        )
        val_f1 = float(
            f1_score(
                validation["labels"],
                validation["probabilities"] >= 0.5,
                zero_division=0,
            )
        )
        if scheduler:
            scheduler.step(val_auprc)
        history["train_loss"].append(loss_sum / max(seen, 1))
        history["train_accuracy"].append(correct / max(seen, 1))
        history["val_loss"].append(validation["loss"])
        history["val_auprc"].append(val_auprc)
        history["val_f1"].append(val_f1)
        print(
            f"Epoch {epoch:02d}/{epochs:02d} | train_loss={history['train_loss'][-1]:.6f} | val_auprc={val_auprc:.6f} | val_f1={val_f1:.6f}"
        )
        if val_auprc > best_auprc:
            best_auprc, best_epoch = val_auprc, epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
    training_seconds = time.time() - training_start
    if best_state is None:
        raise RuntimeError("No MLP checkpoint selected")
    model.load_state_dict(
        {name: value.to(device) for name, value in best_state.items()}
    )
    torch.save(
        {"model_state": best_state, "run_id": run_id, "config": cfg},
        run_dir / "best_checkpoint.pt",
    )
    validation = infer(model, val_loader, device, evaluation_seed)
    threshold, threshold_report = find_optimal_threshold(
        validation["probabilities"],
        validation["labels"],
        float(cfg["threshold_calibration"]["min_precision_constraint"]),
    )
    test_loader = make_loader(
        test_df, features, batch_size, False, evaluation_seed + 10_000
    )
    test = infer(model, test_loader, device, evaluation_seed + 10_000)
    metrics = evaluate_predictions(test["probabilities"], test["labels"], threshold)
    statistics["frozen_test_evaluation"] = {
        "evaluated_after_checkpoint_and_threshold_freeze": True,
        "num_transactions": len(test["labels"]),
        "num_fraud": int(test["labels"].sum()),
        "fraud_rate_pct": float(100 * test["labels"].mean()),
    }
    write_json(run_dir / "window_statistics.json", statistics)
    predictions = pd.concat(
        [
            prediction_frame(
                val_df, validation["probabilities"], "validation", threshold
            ),
            prediction_frame(test_df, test["probabilities"], "test", threshold),
        ],
        ignore_index=True,
    )
    predictions.to_csv(run_dir / "predictions.csv.gz", index=False, compression="gzip")
    np.savez_compressed(
        run_dir / "predictions.npz",
        validation_transaction_id=val_df["transaction_id"].to_numpy(),
        validation_probabilities=validation["probabilities"],
        validation_labels=validation["labels"],
        test_transaction_id=test_df["transaction_id"].to_numpy(),
        test_probabilities=test["probabilities"],
        test_labels=test["labels"],
        optimal_threshold=threshold,
    )
    write_json(run_dir / "training_history.json", history)
    total_seconds = time.time() - started
    output = {
        "experiment_id": "EXP21E",
        "version": version,
        "run_id": run_id,
        "stage": "tabular_mlp_baseline",
        "scientific_status": "smoke_only_not_final"
        if args.smoke
        else "v7_baseline_candidate",
        "model_type": "end_to_end_tabular_mlp",
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "feature_columns": features,
        "time_window": time_cfg,
        "threshold_calibration": threshold_report,
        **metrics,
        "hyperparameters": {
            "hidden_channels": int(cfg["model"]["hidden_channels"]),
            "layers": 2,
            "dropout": float(cfg["model"]["dropout"]),
            "parameter_count": parameter_count,
            "epochs_budget": epochs,
            "best_epoch": best_epoch,
            "best_validation_auprc": best_auprc,
            "learning_rate": float(training_cfg["learning_rate"]),
            "weight_decay": float(training_cfg["weight_decay"]),
            "loss_weight_strategy": training_cfg["loss_weight_strategy"],
            "positive_weight": float(positive_weight),
        },
        "runtime_seconds": {
            "data_preparation": data_seconds,
            "training": training_seconds,
            "validation_inference": validation["seconds"],
            "test_inference": test["seconds"],
            "total": total_seconds,
        },
        "throughput_rows_per_second": {
            "validation": validation["throughput_rows_per_second"],
            "test": test["throughput_rows_per_second"],
        },
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated())
        if torch.cuda.is_available()
        else None,
        "peak_cpu_memory_bytes": peak_cpu_memory_bytes(),
    }
    write_json(run_dir / "metrics.json", output)
    manifest.update(
        {
            "status": "complete",
            "best_epoch": best_epoch,
            "best_validation_auprc": best_auprc,
            "test_evaluations": 1,
            "total_runtime_seconds": total_seconds,
        }
    )
    write_json(run_dir / "manifest.json", manifest)
    print(
        json.dumps(
            {"status": "complete", "run_id": run_id, "run_dir": str(run_dir)}, indent=2
        )
    )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze_results.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd


SEEDS = (42, 100, 2026, 3407, 8713)
METRICS = ("auprc", "roc_auc", "f1_score", "recall", "precision", "gmean")


def bootstrap_ci(values: np.ndarray, seed: int = 2107) -> list[float]:
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(20_000, len(values)), replace=True).mean(axis=1)
    return [float(value) for value in np.quantile(draws, [0.025, 0.975])]


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Summarize the five EXP21E-v7 MLP seeds"
    )
    parser.add_argument("--result-dir", required=True)
    args = parser.parse_args()
    root = Path(args.result_dir)
    runs: dict[int, tuple[Path, dict]] = {}
    for metrics_path in sorted(root.glob("exp21e_v7_*/metrics.json")):
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        seed = int(metrics["seed"])
        if seed in runs:
            raise RuntimeError(f"Duplicate completed v7 run for seed {seed}")
        runs[seed] = (metrics_path.parent, metrics)
    missing = sorted(set(SEEDS) - set(runs))
    extra = sorted(set(runs) - set(SEEDS))
    if missing or extra:
        raise RuntimeError(f"Expected seeds {SEEDS}; missing={missing}, extra={extra}")

    reference_ids = None
    reference_labels = None
    rows = []
    for seed in SEEDS:
        run_dir, metrics = runs[seed]
        predictions = np.load(run_dir / "predictions.npz")
        ids = predictions["test_transaction_id"]
        labels = predictions["test_labels"]
        if reference_ids is None:
            reference_ids, reference_labels = ids, labels
        elif not np.array_equal(ids, reference_ids) or not np.array_equal(
            labels, reference_labels
        ):
            raise RuntimeError(f"Prediction alignment failed for seed {seed}")
        independent = metrics["metrics_threshold_independent"]
        optimal = metrics["metrics_optimal_threshold"]
        rows.append(
            {
                "seed": seed,
                "run_id": metrics["run_id"],
                "best_epoch": metrics["hyperparameters"]["best_epoch"],
                "validation_auprc": metrics["hyperparameters"]["best_validation_auprc"],
                "auprc": independent["auprc"],
                "roc_auc": independent["roc_auc"],
                "f1_score": optimal["f1_score"],
                "recall": optimal["recall"],
                "precision": optimal["precision"],
                "gmean": optimal["gmean"],
                "alerts_per_captured_fraud": optimal["alerts_per_captured_fraud"],
                "runtime_seconds": metrics["runtime_seconds"]["total"],
            }
        )
    frame = pd.DataFrame(rows)
    frame.to_csv(root / "v7_run_summary.csv", index=False)
    summary = {
        "experiment": "EXP21E-v7",
        "model": "end_to_end_tabular_mlp",
        "seeds": list(SEEDS),
        "n_runs": len(frame),
        "alignment_verified": True,
        "selection_rule": "No seed selection; report the complete predeclared five-seed distribution.",
        "metrics": {},
    }
    for metric in METRICS:
        values = frame[metric].to_numpy(dtype=float)
        summary["metrics"][metric] = {
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)),
            "bootstrap_95_ci_of_seed_mean": bootstrap_ci(values),
            "values": [float(value) for value in values],
        }
    (root / "v7_statistical_report.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_all.py
from __future__ import annotations

import argparse
import subprocess
import sys
from pathlib import Path

import yaml


SEEDS = (42, 100, 2026, 3407, 8713)


def run(command: list[str]) -> None:
    print(f"\n>>> {' '.join(command)}", flush=True)
    subprocess.run(command, check=True)


def resolve_config(explicit: str | None) -> Path:
    if explicit:
        return Path(explicit)
    if Path("/kaggle/input").exists():
        return Path("config.kaggle.yaml")
    if Path("/content").exists():
        return Path("config.colab.yaml")
    return Path("config.reference.yaml")


def resolve_kaggle_dataset(config_path: Path) -> None:
    if not Path("/kaggle/input").exists():
        return
    matches = sorted(Path("/kaggle/input").rglob("credit_card_transactions-ibm_v2.csv"))
    if not matches:
        raise FileNotFoundError(
            "credit_card_transactions-ibm_v2.csv was not found under /kaggle/input"
        )
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    config["data"]["transactions"] = str(matches[0])
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Run the five EXP21E-v7 tabular MLP seeds"
    )
    parser.add_argument("--config", default=None)
    args = parser.parse_args()
    config_path = resolve_config(args.config)
    resolve_kaggle_dataset(config_path)
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    result_dir = Path(config["output"]["result_dir"]) / "v7"
    result_dir.mkdir(parents=True, exist_ok=True)
    run(
        [
            sys.executable,
            "tabular_validation.py",
            "--output",
            str(result_dir / "tabular_validation_report.json"),
        ]
    )
    for seed in SEEDS:
        run(
            [
                sys.executable,
                "main.py",
                "--config",
                str(config_path),
                "--seed",
                str(seed),
            ]
        )
    run([sys.executable, "analyze_results.py", "--result-dir", str(result_dir)])


if __name__ == "__main__":
    main()


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, 'run_all.py'], check=True)
